In [ ]:
import torch
torch.cuda.is_available()


False

In [ ]:
import json
import pandas as pd

In [ ]:
def load_and_flatten_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            ex = json.loads(line)
            text = ex["input"]
            outputs = ex.get("output", [])
            for ann in outputs:
                # one row per (aspect, emotion)
                rows.append({
                    "text": text,
                    "aspect": ann["aspect"],
                    "emotion": ann["emotion"],
                    "polarity": ann.get("polarity", None),
                    "line_no": line_no
                })
    return pd.DataFrame(rows)

In [ ]:
train_df = load_and_flatten_jsonl("gpt_output_train.jsonl")
val_df   = load_and_flatten_jsonl("gpt_output_validation.jsonl")
test_df  = load_and_flatten_jsonl("gpt_output_test.jsonl")

In [ ]:
def make_input_text(df):
    df = df.copy()
    df["input_text"] = df.apply(
        lambda r: f"Review: {r['text']} [SEP] Aspect: {r['aspect']}",
        axis=1
    )
    return df

train_df = make_input_text(train_df)
val_df   = make_input_text(val_df)
test_df  = make_input_text(test_df)


In [ ]:
sorted(train_df["emotion"].unique())


['admiration',
 'annoyance',
 'disappointment',
 'disgust',
 'gratitude',
 'mentioned_only',
 'mixed_emotions',
 'neutral',
 'null',
 'satisfaction']

In [ ]:
labels = [
    'admiration',
 'annoyance',
 'disappointment',
 'disgust',
 'gratitude',
 'mentioned_only',
 'mixed_emotions',
 'neutral',
 'satisfaction'
]

label2id = {lab: i for i, lab in enumerate(labels)}
id2label = {i: lab for lab, i in label2id.items()}


In [ ]:
missing = set(train_df["emotion"].unique()) - set(labels)
print("Labels in data but not in labels list:", missing)


Labels in data but not in labels list: {'null'}


In [ ]:
# Remove 'null' From All Files
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(name, (df["emotion"] == "null").sum())


train 2
val 2
test 0


In [ ]:
def drop_null_emotions(df):
    return df[df["emotion"] != "null"].reset_index(drop=True)

train_df = drop_null_emotions(train_df)
val_df   = drop_null_emotions(val_df)
test_df  = drop_null_emotions(test_df)


In [ ]:
missing = set(train_df["emotion"].unique()) - set(labels)
print("Labels in data but not in labels list:", missing)


Labels in data but not in labels list: set()


In [ ]:
train_df["emotion"].value_counts()


,count
emotion,
mentioned_only,2536
annoyance,1347
satisfaction,1089
admiration,834
disappointment,699
neutral,539
disgust,30
mixed_emotions,12
gratitude,2


In [ ]:
from transformers import RobertaTokenizer, RobertaForSequenceClassification

tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=9,
    id2label=id2label,
    label2id=label2id
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
def tokenize(batch):
    return tokenizer(
        batch["input_text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )
